# Data collection for the year 2021

In [1]:
import cocopp
dsl = cocopp.load("bbob/2021/*")

  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2021/SHADE-LM-POP4-to-10_Okulewicz.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2021\SHADE-LM-POP4-to-10_Okulewicz.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2021/SHADE-LM_Okulewicz.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2021\SHADE-LM_Okulewicz.tgz
    archive extracted to folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2021\.extracted_SHADE-LM-POP4-to-10_Okulewicz ...
    archive extracted to folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2021\.extracted_SHADE-LM_Okulewicz ...
  Data consistent according to consistency_check() in pproc.DataSet


In [2]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=6 @ 1e-08)
dim= 2, F 2 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=232 @ 1e-08)
dim= 2, F 3 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=651 @ 1e-08)
dim= 2, F 4 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=1.17e+03 @ 1e-08)
dim= 2, F 5 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=4 @ 1e-08)
dim= 2, F 6 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=622 @ 1e-08)
dim= 2, F 7 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=264 @ 1e-08)
dim= 2, F 8 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=608 @ 1e-08)
dim= 2, F 9 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=457 @ 1e-08)
dim= 2, F10 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=613 @ 1e-08)
dim= 2, F11 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=515 @ 1e-08)
dim= 2, F12 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=1.89e+03 @ 1e-08)
dim= 2, F13 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=518 @ 1e-08)
dim= 2, F14 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=505 @ 1e-08)
dim= 2, F15 -> SHADE-LM-POP4-to-10_Okulewicz  (ERT=915 @ 1e-08)
dim= 2, F16 -> SHADE-LM-POP4-to-10

In [3]:
from collections import Counter, defaultdict

In [4]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

SHADE-LM-POP4-to-10_Okulewicz: 73
SHADE-LM_Okulewicz: 40


In [5]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'SHADE-LM-POP4-to-10_Okulewicz',
 3: 'SHADE-LM-POP4-to-10_Okulewicz',
 5: 'SHADE-LM-POP4-to-10_Okulewicz',
 10: 'SHADE-LM_Okulewicz',
 20: 'SHADE-LM-POP4-to-10_Okulewicz'}

In [6]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20]
    dimension  function_id        target                 best_algorithm  \
0           2            1  1.000000e-08  SHADE-LM-POP4-to-10_Okulewicz   
1           2            1  1.000000e-05  SHADE-LM-POP4-to-10_Okulewicz   
2           2            1  1.000000e-03  SHADE-LM-POP4-to-10_Okulewicz   
3           2            1  1.000000e-02  SHADE-LM-POP4-to-10_Okulewicz   
4           2            1  1.000000e-01  SHADE-LM-POP4-to-10_Okulewicz   
5           2            2  1.000000e-08  SHADE-LM-POP4-to-10_Okulewicz   
6           2            2  1.000000e-05  SHADE-LM-POP4-to-10_Okulewicz   
7           2            2  1.000000e-03  SHADE-LM-POP4-to-10_Okulewicz   
8           2            2  1.000000e-02  SHADE-LM-POP4-to-10_Okulewicz   
9           2            2  1.000000e-01  SHADE-LM-POP4-to-10_Okulewicz   
10          2            3  1.000000e-08  SHADE-LM-POP4-to-10_Okulewicz   
11          2            3  1.000000e-05  SHADE-LM-PO

In [7]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2021.csv", index=False)
